In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json

In [ ]:
with open("./config.json") as f:
    config = json.load(f)
    collection_start_date = int(config["collection_start_date"])
    collection_end_date = int(config["collection_end_date"])
    root_output_dir_path = config["root_output_dir_path"]
    output_dir = root_output_dir_path + "/embeds"


In [ ]:
# collate embeddings
df_list = []
batch_dir = root_output_dir_path + "/modernbert_embed"
batch_files = os.listdir(batch_dir)
sorted_batch_files = sorted(batch_files, key=lambda x: int(re.findall(r"\d+", x)[0]))
for f in sorted_batch_files:
    df = pd.read_pickle(batch_dir + "/" + f)
    df_list.append(df)
text_embeds = pd.concat(df_list, ignore_index=True)
text_embeds.to_pickle(root_output_dir_path + "/text_embedding.pkl")


In [ ]:
# collate submission embeddings
df_list = []
batch_dir = root_output_dir_path + "/modernbert_submission_embed"
batch_files = os.listdir(batch_dir)
sorted_batch_files = sorted(batch_files, key=lambda x: int(re.findall(r"\d+", x)[0]))
for f in sorted_batch_files:
    df = pd.read_pickle(batch_dir + "/" + f)
    df_list.append(df)
submission_embeds = pd.concat(df_list, ignore_index=True)
submission_embeds.to_pickle(root_output_dir_path + "/submission_text_embedding.pkl")

In [ ]:
final_df = pd.concat([text_embeds,submission_embeds], ignore_index=True)

grouped = final_df.groupby("screen_name")["embedding"].apply(lambda x: np.mean(x.values))
screen_name_list = grouped.index.tolist()
user_embeds_list = grouped.values.tolist()
user_embedding_df = pd.DataFrame({
    "screen_name": screen_name_list,
    "user_embedding": user_embeds_list
})
user_embedding_df.to_pickle(root_output_dir_path + "/user_embedding.pkl")